In [7]:
import torch 
import math 
import torch.nn as nn 
import torch.nn.functional as F

In [8]:
class CausalMultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_query = nn.Linear(d_model, d_model, bias=False)
        self.W_key = nn.Linear(d_model, d_model, bias=False)
        self.W_value = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):

        batch_size, num_tokens, _ = x.shape

        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        Q = Q.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / (self.head_dim ** 0.5)

        causal_mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device), diagonal=1).bool()

        scores = scores.masked_fill(causal_mask, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)

        context = attention_weights @ V

        context = context.transpose(1, 2).contiguous().view(
            batch_size,
            num_tokens,
            self.d_model
        )
        
        output = self.out_proj(context)

        return output, attention_weights


In [9]:
batch_size = 2
num_tokens = 8
d_model = 256
num_heads = 8

x = torch.randn(batch_size, num_tokens, d_model)

attention = CausalMultiHeadSelfAttention(
    d_model=d_model,
    num_heads=num_heads
)

output, attention_weights = attention(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention shape:", attention_weights.shape)

Input shape: torch.Size([2, 8, 256])
Output shape: torch.Size([2, 8, 256])
Attention shape: torch.Size([2, 8, 8, 8])
